# Virtual Screening & ADMET Prediction

**Tier 3 — Applied Bioinformatics | Module 29 · Notebook 3**

*Prerequisites: Notebook 2 (QSAR Modeling)*

---

**By the end of this notebook you will be able to:**
1. Explain ligand-based vs structure-based virtual screening strategies
2. Prepare a protein structure for docking (protonation, grid generation)
3. Run AutoDock Vina to dock a small molecule library
4. Filter compounds by ADMET properties using SwissADME and pkCSM
5. Prioritize hits from combined docking + QSAR + ADMET scores


**Key resources:**
- [AutoDock Vina documentation](https://vina.scripps.edu/)
- [SwissADME web tool](http://www.swissadme.ch/)
- [TeachOpenCADD docking talktorial](https://github.com/volkamerlab/teachopencadd)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This workflow maps to production-style applied bioinformatics: pipeline design, model evaluation, and biological interpretation for decision-making.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Pipeline outputs are context-dependent: QC failures upstream can invalidate downstream interpretation.
- Batch effects and confounders can dominate signal if not controlled explicitly.
- Use uncertainty-aware decisions: rank candidates and keep rationale for every threshold.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: QSAR Modeling](./02_qsar_modeling.ipynb) | [Next: Graph Neural Networks →](./04_molecular_gnn.ipynb)

---

## 1. Virtual Screening Overview

> Diagram of drug discovery pipeline: target → hit → lead → candidate. Two VS strategies: ligand-based (pharmacophore, similarity, QSAR) vs structure-based (molecular docking). Throughput/accuracy trade-offs.

## 2. Protein Structure Preparation

> Download EGFR crystal structure (PDB: 1IEP). Remove water/ligands. Add hydrogen atoms with Open Babel or PDBFixer. Define docking box around the ATP binding site.

In [ ]:
# Example: Download and prepare protein structure
# from Bio.PDB import PDBParser, PDBIO, Select
# import py3Dmol

# !wget -q https://files.rcsb.org/download/1IEP.pdb
# !pdbfixer 1IEP.pdb --output 1IEP_fixed.pdb --add-hydrogens --remove-heterogens

# Example: Visualize structure with py3Dmol
# view = py3Dmol.view()
# view.addModel(open('1IEP_fixed.pdb').read(), 'pdb')
# view.setStyle({'cartoon': {'color': 'spectrum'}})
# view.show()

## 3. Molecular Docking with AutoDock Vina

> Prepare ligand SDF to PDBQT with meeko. Set docking box (center_x/y/z, size_x/y/z) from known active site. Run AutoDock Vina. Parse best docking score.

In [ ]:
# Example: Prepare and dock ligand
# !mk_prepare_ligand.py -i gefitinib.sdf -o gefitinib.pdbqt

# vina_config = """
# receptor = 1IEP_fixed.pdbqt
# ligand = gefitinib.pdbqt
# center_x = 22.5; center_y = 5.0; center_z = 18.0
# size_x = 20; size_y = 20; size_z = 20
# exhaustiveness = 8
# """
# with open('vina_config.txt', 'w') as f: f.write(vina_config)
# !vina --config vina_config.txt --out gefitinib_docked.pdbqt --log docking.log
# !grep 'REMARK VINA RESULT' gefitinib_docked.pdbqt | head

## 4. ADMET Property Prediction

> Compute ADMET properties: Lipinski Ro5, LogS (solubility), Pgp substrate, CYP inhibition, hERG toxicity. Use SwissADME API or DeepChem ADMET models.

In [ ]:
# Example: ADMET with DeepChem
# import deepchem as dc
# smiles_list = ['CC1=C(C=C(C=C1)NC2=NC=CC(=N2)N3CCN(CC3)C)NC4=NC=CC(=N4)Cl']  # Imatinib
# featurizer = dc.feat.CircularFingerprint(size=2048)
# X = featurizer.featurize(smiles_list)
# # Example: Use pre-trained Tox21 model for toxicity prediction

## 5. Hit Prioritization

> Combine docking score, QSAR pIC50 prediction, and ADMET pass/fail into a composite score. Rank library of 100 compounds. Visualize top 10 structures with their scores.

In [ ]:
import pandas as pd

# Example: Composite scoring
# hits = pd.DataFrame({
#     'SMILES': smiles_library,
#     'vina_score': docking_scores,
#     'qsar_pIC50': qsar_predictions,
#     'admet_pass': admet_flags
# })
# hits_filtered = hits[hits['admet_pass']]
# hits_filtered['composite'] = -hits_filtered['vina_score'] + hits_filtered['qsar_pIC50']
# top10 = hits_filtered.nlargest(10, 'composite')
# print(top10[['SMILES', 'vina_score', 'qsar_pIC50', 'composite']])

## Summary

> Recap VS pipeline: library preparation → docking → ADMET filter → QSAR scoring → prioritization. Discuss limitations of rigid docking (induced fit, water molecules). Link to GNN notebook for end-to-end learned scoring.

---

[← Previous: QSAR Modeling](./02_qsar_modeling.ipynb) | [Next: Graph Neural Networks →](./04_molecular_gnn.ipynb)

---